# RQ2 diagnostic alignment

Original notebook used to calculate skill-wise diagnostic correlations. Edit only the configuration paths before running.


## Path configuration

The analysis logic below is unchanged from the research notebook. Theta CSVs
are generated model outputs, not raw input data: train the selected model and
run its CP2 and CP4 diagnosis commands first. Then replace every
`REPLACE_WITH_...` value with the named generated output or authorized target
file. Paths may be absolute or repository-root-relative. Analysis outputs are
written under the repository's `outputs/` directory.


# Experiment 2 — S1 explicit theta: pooled Spearman


1. `CP2 theta → Mid score`
2. `CP4 theta → Final score`



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display

# Paths are explicit so this notebook can be run from any working directory.
# USER CONFIGURATION: replace these six input paths before running.
THETA_CP2_PATH = Path("REPLACE_WITH_CP2_THETA_CSV")
THETA_CP4_PATH = Path("REPLACE_WITH_CP4_THETA_CSV")
TARGET_MID_PATH = Path("REPLACE_WITH_MID_SCORE_CSV")
TARGET_FINAL_PATH = Path("REPLACE_WITH_FINAL_SCORE_CSV")
Q_MID_PATH = Path("REPLACE_WITH_MID_Q_MATRIX_CSV")
Q_FINAL_PATH = Path("REPLACE_WITH_FINAL_Q_MATRIX_CSV")

# This output path normally does not need to be changed.
OUTPUT_DIR = Path("outputs/RQ2_Diagnostic_Alignment/without_attempt/corr_theta_explicit_S1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECIFIC_SCORE_MODE = "mean"  # choose "mean" or "sum"
MIN_N_PER_SKILL = 3

INPUT_PATHS = [
    THETA_CP2_PATH, THETA_CP4_PATH,
    TARGET_MID_PATH, TARGET_FINAL_PATH,
    Q_MID_PATH, Q_FINAL_PATH,
]
missing = [str(path) for path in INPUT_PATHS if not path.exists()]
if missing:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(missing))

print("Split:", SPLIT)
print("Theta variant:", THETA_VARIANT)
print("CP2 theta:", THETA_CP2_PATH)
print("CP4 theta:", THETA_CP4_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    return df


def find_student_column(df):
    candidates = [
        "user_id", "student_id", "cpstudent", "student", "id",
        "เด็ก", "เด็ก/ข้อ",
    ]
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"Cannot find a student id column. Columns={df.columns.tolist()}")


def load_theta_long(path, target_exam):
    """Read K0..K17 theta and return one row per student/skill."""
    df = clean_columns(pd.read_csv(path))
    student_col = find_student_column(df)
    skill_cols = [col for col in df.columns if col.startswith("K")]
    if not skill_cols:
        raise ValueError(f"No K* columns in {path}")

    out = df[[student_col] + skill_cols].rename(
        columns={student_col: "student_id"}
    )
    out["student_id"] = pd.to_numeric(out["student_id"], errors="coerce")
    out = out.dropna(subset=["student_id"]).copy()
    out["student_id"] = out["student_id"].astype(int)
    out = out.melt(
        id_vars="student_id",
        value_vars=skill_cols,
        var_name="skill",
        value_name="theta",
    )
    out["skill"] = pd.to_numeric(
        out["skill"].str.replace("K", "", regex=False), errors="coerce"
    )
    out["theta"] = pd.to_numeric(out["theta"], errors="coerce")
    out = out.dropna(subset=["skill", "theta"]).copy()
    out["skill"] = out["skill"].astype(int)
    out["target_exam"] = target_exam
    return out[["student_id", "target_exam", "skill", "theta"]]


def load_score_long(path, target_exam):
    """Support both long (item_id, score) and wide response files."""
    df = clean_columns(pd.read_csv(path))
    student_col = find_student_column(df)

    if {"item_id", "score"}.issubset(df.columns):
        out = df[[student_col, "item_id", "score"]].rename(
            columns={student_col: "student_id"}
        )
    else:
        item_cols = [col for col in df.columns if str(col).isdigit()]
        if not item_cols:
            raise ValueError(f"No item response columns in {path}")
        out = df[[student_col] + item_cols].rename(
            columns={student_col: "student_id"}
        ).melt(
            id_vars="student_id",
            value_vars=item_cols,
            var_name="item_id",
            value_name="score",
        )

    for column in ["student_id", "item_id", "score"]:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    out = out.dropna(subset=["student_id", "item_id", "score"]).copy()
    out["student_id"] = out["student_id"].astype(int)
    out["item_id"] = out["item_id"].astype(int)
    out["target_exam"] = target_exam
    return out[["student_id", "target_exam", "item_id", "score"]]


def load_q_long(path, target_exam):
    """Convert a Q-matrix into active target_exam/item/skill links."""
    df = clean_columns(pd.read_csv(path))
    if "item_id" not in df.columns:
        raise ValueError(f"Q-matrix {path} has no item_id column")
    skill_cols = [col for col in df.columns if col.startswith("K")]
    q_long = df[["item_id"] + skill_cols].melt(
        id_vars="item_id",
        value_vars=skill_cols,
        var_name="skill",
        value_name="required",
    )
    q_long["item_id"] = pd.to_numeric(q_long["item_id"], errors="coerce")
    q_long["required"] = pd.to_numeric(q_long["required"], errors="coerce")
    q_long["skill"] = pd.to_numeric(
        q_long["skill"].str.replace("K", "", regex=False), errors="coerce"
    )
    q_long = q_long.loc[q_long["required"].fillna(0) > 0].dropna(
        subset=["item_id", "skill"]
    ).copy()
    q_long["item_id"] = q_long["item_id"].astype(int)
    q_long["skill"] = q_long["skill"].astype(int)
    q_long["target_exam"] = target_exam
    return q_long[["target_exam", "item_id", "skill"]]


In [ ]:
# Recompute the complete analysis table from raw inputs in this notebook.
theta_all = pd.concat(
    [
        load_theta_long(THETA_CP2_PATH, "Mid"),
        load_theta_long(THETA_CP4_PATH, "Final"),
    ],
    ignore_index=True,
)
score_all = pd.concat(
    [
        load_score_long(TARGET_MID_PATH, "Mid"),
        load_score_long(TARGET_FINAL_PATH, "Final"),
    ],
    ignore_index=True,
)
q_all = pd.concat(
    [
        load_q_long(Q_MID_PATH, "Mid"),
        load_q_long(Q_FINAL_PATH, "Final"),
    ],
    ignore_index=True,
)

# Attach each response to every skill required by its item.
score_skill = score_all.merge(
    q_all,
    on=["target_exam", "item_id"],
    how="inner",
    validate="m:m",
)

group_keys = ["student_id", "target_exam", "skill"]
if SPECIFIC_SCORE_MODE == "mean":
    specific = score_skill.groupby(group_keys, as_index=False)["score"].mean()
elif SPECIFIC_SCORE_MODE == "sum":
    specific = score_skill.groupby(group_keys, as_index=False)["score"].sum()
else:
    raise ValueError("SPECIFIC_SCORE_MODE must be 'mean' or 'sum'")
specific = specific.rename(columns={"score": "specific_score"})

related = (
    score_skill.groupby(group_keys)["item_id"]
    .apply(lambda values: ",".join(map(str, sorted(set(values)))))
    .rename("related_items_used")
    .reset_index()
)
specific = specific.merge(related, on=group_keys, how="left", validate="1:1")

analysis_df = theta_all.merge(
    specific,
    on=group_keys,
    how="left",
    validate="1:1",
).sort_values(["target_exam", "skill", "student_id"]).reset_index(drop=True)

print("theta rows:", len(theta_all))
print("score rows:", len(score_all))
print("analysis rows:", len(analysis_df))
print("students by target exam:")
display(analysis_df.groupby("target_exam")["student_id"].nunique().rename("n_students"))
display(analysis_df.head(12))


In [ ]:
def compute_spearman_by_skill(data, group_label):
    """Compute Spearman correlation independently for every theta skill."""
    rows = []
    all_skills = sorted(theta_all["skill"].unique().tolist())
    for skill in all_skills:
        g = data.loc[
            data["skill"].eq(skill), ["theta", "specific_score"]
        ].dropna()
        n = len(g)
        theta_unique = g["theta"].nunique()
        score_unique = g["specific_score"].nunique()

        if n < MIN_N_PER_SKILL:
            rho, p_value, status = np.nan, np.nan, f"skip: n<{MIN_N_PER_SKILL}"
        elif theta_unique < 2:
            rho, p_value, status = np.nan, np.nan, "skip: theta constant"
        elif score_unique < 2:
            rho, p_value, status = np.nan, np.nan, "skip: specific_score constant"
        else:
            rho, p_value = spearmanr(g["theta"], g["specific_score"])
            status = "ok"

        rows.append({
            "group": group_label,
            "skill": int(skill),
            "n_students": int(n),
            "theta_unique": int(theta_unique),
            "specific_score_unique": int(score_unique),
            "spearman_rho": rho,
            "p_value": p_value,
            "status": status,
        })
    return pd.DataFrame(rows)


def summarize_spearman(results):
    valid = results.loc[
        results["status"].eq("ok") & results["spearman_rho"].notna()
    ]
    return {
        "group": results["group"].iloc[0],
        "n_skills_total": int(len(results)),
        "n_skills_valid": int(len(valid)),
        "mean_spearman": valid["spearman_rho"].mean(),
        "median_spearman": valid["spearman_rho"].median(),
        "min_spearman": valid["spearman_rho"].min(),
        "max_spearman": valid["spearman_rho"].max(),
    }


In [ ]:
pooled_results_df = compute_spearman_by_skill(analysis_df, "Pooled")
pooled_summary_df = pd.DataFrame([summarize_spearman(pooled_results_df)])

print(f"Pooled correlation: {SPLIT} / {THETA_VARIANT}")
display(pooled_results_df)
display(pooled_summary_df)


In [ ]:
analysis_path = OUTPUT_DIR / "pooled_theta_specific_score_table.csv"
results_path = OUTPUT_DIR / "pooled_spearman_by_skill.csv"
summary_path = OUTPUT_DIR / "pooled_spearman_summary.csv"

analysis_df.to_csv(analysis_path, index=False, encoding="utf-8-sig")
pooled_results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
pooled_summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", analysis_path)
print(" -", results_path)
print(" -", summary_path)
